<a href="https://colab.research.google.com/github/Nivedhitha-Arasu/GB885-Final-Project-Thirunavukkarasu-N/blob/main/GB885_Final_Project_Thirunavukkarasu_N.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Business Case - Analyst for a renowned sportswear and footwear brand

**Nivedhitha Thirunavukkarasu**

##Step 1. Understand the Problem

Objective: Analyze sales data for trends and insights that will help company leadership understand the market and identify opportunities for growth.

For example:

*   You may want to look for trends or insights in seasonality, retailers, locations, or sales methods.

In addition, she has asked you to answer the following business questions,

*   What product category (product) had the highest sales (in dollars)   in 2021? How much did it sell?
*   What state had the highest sales (in dollars) of women's products in 2021? How much was it?
*   What state had the highest sales (in dollars) of men's products in 2021? How much was it?
*   What retailer purchased the most units in 2021? In 2020?

In [ ]:
#Import Python Libraries

import pandas as pd
import numpy as np

##Step 2. Data Acquisition and Consolidation

In [ ]:
#Import Data

#Products Data (pipe separated)
products_data_df = pd.read_csv('https://raw.githubusercontent.com/Nivedhitha-Arasu/GB885-Final-Project-Thirunavukkarasu-N/refs/heads/main/TABLE_PRODUCTS_885.csv', sep='|')

#Retailer Data (comma separated)
retailer_data_df = pd.read_csv('https://raw.githubusercontent.com/Nivedhitha-Arasu/GB885-Final-Project-Thirunavukkarasu-N/refs/heads/main/TABLE_RETAILER_885.csv', sep = ',')

#Sales Data (comma separated)
sales_data_df = pd.read_csv('https://raw.githubusercontent.com/Nivedhitha-Arasu/GB885-Final-Project-Thirunavukkarasu-N/refs/heads/main/TABLE_SALES_885.csv', sep = ',')

In [ ]:
#Preview Products data
products_data_df.head()

In [ ]:
#Preview Retailer data
retailer_data_df.head()

In [ ]:
#Preview Sales data
sales_data_df.head()

###**Consolidate three dataframes**

In [ ]:
#merge products data and sales data based on product_id

product_sales_df = pd.merge(sales_data_df, products_data_df, on='PRODUCT_ID', how='left')

#check our work
product_sales_df.head()

In [ ]:
#merge new product_sales data with retailer data based on retailer_id

full_sales_df = pd.merge(product_sales_df, retailer_data_df, on='RETAILER_ID', how='left')

#check our work
full_sales_df.head()

##Step 3. Inspect the Data

Look For:
* Unwanted Observations
* Unwanted Features
* Incorrect Data Formats or DataTypes
* Duplicate Values
* Missing Values
* Erroneous Values
* Outliers

###**Unwanted Observations:** filtering data to answer the required business questions

In [ ]:
#filter full data to only include observations from 2021 since most of the business questions are based on 2021

#create the filter
year_filter = full_sales_df['YEAR'] == '2021'
#apply the filter
sales_2021_df = full_sales_df[year_filter]

###**Unwanted features**

In [ ]:
#not removing any for now as every column seems to be required for analysis

###**Incorrect Data Formats or DataTypes**

In [ ]:
full_sales_df.info()

In [ ]:
#Convert units sold column to integer

full_sales_df['UNITS_SOLD'] = pd.to_numeric(full_sales_df['UNITS_SOLD'], errors='coerce').fillna(0).astype(int)

In [ ]:
# Convert column 'invoice_date' to datetime

full_sales_df['INVOICE_DATE'] = pd.to_datetime(full_sales_df['INVOICE_DATE'])

###**Duplicate Values**

In [ ]:
full_sales_df.duplicated().sum()

###**Missing Values**

In [ ]:
full_sales_df.shape

In [ ]:
#Null Values (saved as, or coerced to, NA)
full_sales_df.isnull().sum()

###**Check for Erroneous Values**

In [ ]:
#check extremes for erroneous data
full_sales_df.describe()

In [ ]:
#list the categorical variables

new_cat_var = list(full_sales_df.select_dtypes(include=['object']).columns)
new_cat_var

In [ ]:
#unique values of each categorical variable

for column in new_cat_var:
    print(column)
    print(full_sales_df[column].unique())

###**Check for Outliers**

In [ ]:
#write a function to calculate IQR and print rows with values that fall outside that IQR

def count_iqr_outliers(df, column):
    # define q1
    q1 = df[column].quantile(0.25)
    # define q3
    q3 = df[column].quantile(0.75)
    # define iqr
    iqr = q3 - q1
    # define outlier thresholds
    l_threshold = q1 - 1.5 * iqr
    u_threshold = q3 + 1.5 * iqr
    # dount outliers
    outliers = (df[column] < l_threshold) | (df[column] > u_threshold)
    # Count the number of True values (outliers)
    return outliers.sum()

In [ ]:
#iterate over the numerical columns of the dataframe:

num_var = list(full_sales_df.select_dtypes(include=['int64', 'float64']).columns)

for column in num_var:
    print(f'{column} : {count_iqr_outliers(full_sales_df, column)}')

##Step 4. Cleaning Data

###**Handle Missing Values**

In [ ]:
#impute zero for price_per_unit missing values

full_sales_df['PRICE_PER_UNIT'] = full_sales_df['PRICE_PER_UNIT'].fillna(0)


In [ ]:
#impute "None" for retailer, region, state, and city missing value

#Define your target columns
columns_to_impute = ['RETAILER', 'REGION', 'STATE', 'CITY']

#Fill missing values with "None"
full_sales_df[columns_to_impute] = full_sales_df[columns_to_impute].fillna("None")

In [ ]:
full_sales_df.isnull().sum()

###**Handle Erroneous Values**

In [ ]:
#Examine Sales method erroneous value

ootlet_count = (full_sales_df['SALES_METHOD'] == 'Ootlet').sum()
print(ootlet_count)

In [ ]:
#replace Sales method erroneous value with Outlet

full_sales_df.replace(to_replace='Ootlet',value='Outlet', inplace = True)

In [ ]:
#Examining the erroneous value in Retailer column

print(full_sales_df['RETAILER'].unique())

In [ ]:
#replace the "Kohl's" value in Retailer column to avoid syntax issues

full_sales_df.replace(to_replace="Kohl's",value='Kohls', inplace = True)

In [ ]:
#check the work

for column in new_cat_var:
    print(column)
    print(full_sales_df[column].unique())

###**Handle Outliers**

In [ ]:
#re-evaluate outliers post cleaning
#list of numerical variables

num_var = list(full_sales_df.select_dtypes(include=['int64', 'float64']).columns)

for column in num_var:
    print(column)
    print(count_iqr_outliers(full_sales_df, column))

###**Feature Engineering**

In [ ]:
# Creating a new column for total sales revenue

# Create 'Sales Revenue' column by multiplying 'Price per unit' and 'Units sold'
full_sales_df['SALES_REVENUE'] = full_sales_df['PRICE_PER_UNIT'] * full_sales_df['UNITS_SOLD']

In [ ]:
full_sales_df.head()

In [ ]:
#categorical feature engineering
#list of categorical features

list(full_sales_df.select_dtypes(include=['object']).columns)

##Step 5. Prepare for Analysis

We can save two versions of the data frame for two purposes:

* A dataframe for descriptive analysis that will include everything we've done up to this point.
* A dataframe that we will prepare further for predictive analysis.

In [ ]:
#create version of dataframe for descriptive and visual analysis

sales_descriptive = full_sales_df.copy()

#create version of dataframe for predictive analysis

sales_predictive = full_sales_df.copy()

####Categorical: One-Hot Encoding

In [ ]:
#list the rest of the categorical variables

new_cat_var = list(sales_predictive.select_dtypes(include=['object']).columns)
new_cat_var

In [ ]:
#unique values of each categorical variable

for column in new_cat_var:
    print(column)
    print(sales_predictive[column].unique())

In [ ]:
# features that need one-hot encoding

select_features = ['SALES_METHOD', 'PRODUCT_NAME', 'REGION', 'RETAILER']

In [ ]:
#create one-hot variables for the select_features columns

sales_predictive = pd.get_dummies(data=sales_predictive,
                                 columns=select_features,
                                 dtype=int)

In [ ]:
#pandas settings

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [ ]:
#check our work

sales_predictive.head()

In [ ]:
#drop columns that are now redundant

sales_predictive.drop(columns=['RETAILER_None', 'REGION_None'], inplace=True)

##Step 6. Analysis

###Descriptive Analysis

In [ ]:
#descriptive statistics

sales_descriptive.describe().T

In [ ]:
sales_descriptive.head()

Descriptive Analysis: GroupBy

In [ ]:
#average price per unit for each product type

sales_descriptive.groupby('PRODUCT_NAME')['PRICE_PER_UNIT'].mean()

In [ ]:
#average units sold for each product type

sales_descriptive.groupby('PRODUCT_NAME')['UNITS_SOLD'].mean()

In [ ]:
#total units sold per sales method

sales_descriptive.groupby('SALES_METHOD')['UNITS_SOLD'].sum().sort_values(ascending=False)

In [ ]:
#total units sold per retailer

sales_descriptive.groupby('RETAILER')['UNITS_SOLD'].sum().sort_values(ascending=False)

Visual Analysis

In [ ]:
#import visualization libraries

import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Create a histogram of Sales Revenue using seaborn

plt.figure(figsize=(10, 6))
sns.histplot(data=sales_descriptive, x='SALES_REVENUE', hue='YEAR', bins=40, kde=True)
plt.title('Histogram of Sales Revenue', fontsize=20)
plt.xlabel('Sales Revenue')
plt.ylabel('Frequency')
plt.show()

###Predictive Analysis

In [ ]:
#Import library

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

In [ ]:
#separate feature and label

#label
Y = sales_predictive['SALES_REVENUE']

#features
X = sales_predictive.drop('SALES_REVENUE', axis=1)

In [ ]:
#Select the model: Linear regression
linreg = LinearRegression()

# Drop non-numeric columns from X before training
X_numeric = X.drop(columns=['RETAILER_ID', 'INVOICE_DATE', 'STATE', 'CITY'])

#train the model
linreg.fit(X_numeric, Y)

In [ ]:
#model r-squared
linreg.score(X_numeric, Y)

###Answering the business questions

In [ ]:
#What product category (product) had the highest sales (in dollars) in 2021? How much did it sell?

#create the filter
year_filter = full_sales_df['YEAR'] == 2021
#apply the filter
sales_2021_df = full_sales_df[year_filter]

# Add up sales for each product category
product_sales = sales_2021_df.groupby("PRODUCT_NAME")["SALES_REVENUE"].sum()

# Find the product with the highest sales
top_product = product_sales.idxmax()
highest_sales = product_sales.max()

print("Product category with the highest sales:", top_product)
print(f"Total sales in 2021: $, {highest_sales/ 1e6:.2f}M")

In [ ]:
#What state had the highest sales (in dollars) of women's products in 2021? How much was it?

items_to_keep = ["Women's Street Footwear", "Women's Athletic Footwear", "Women's Apparel"]

# Filter for women's products in 2021
women_2021 = sales_2021_df[
    (sales_2021_df["PRODUCT_NAME"].isin(items_to_keep))
]

# Calculate total sales by state
sales_by_state = women_2021.groupby("STATE")["SALES_REVENUE"].sum()

# Find the state with the highest sales
highest_state = sales_by_state.idxmax()
highest_sales = sales_by_state.max()

print("State with highest women's product sales:", highest_state)
print(f"Sales: ${highest_sales/ 1e6:.2f}M")

In [ ]:
#What state had the highest sales (in dollars) of men's products in 2021? How much was it?

items_to_keep = ["Men's Street Footwear", "Men's Athletic Footwear", "Men's Apparel"]

# Filter for men's products in 2021
men_2021 = sales_2021_df[
    (sales_2021_df["PRODUCT_NAME"].isin(items_to_keep))
]

# Calculate total sales by state
sales_by_state = men_2021.groupby("STATE")["SALES_REVENUE"].sum()

# Find the state with the highest sales
highest_state = sales_by_state.idxmax()
highest_sales = sales_by_state.max()

print("State with highest men's product sales:", highest_state)
print(f"Sales: ${highest_sales/ 1e6:.2f}M")

In [ ]:
#What retailer purchased the most units in 2021? In 2020?

# Filter for 2020 and 2021, then sum units by retailer
units_by_retailer = (
    full_sales_df[full_sales_df["YEAR"].isin([2020, 2021])]
    .groupby(["YEAR", "RETAILER"])["UNITS_SOLD"]
    .sum()
    .reset_index()
)

# Find the retailer with the most units for each year
top_retailers = (
    units_by_retailer.loc[
        units_by_retailer.groupby("YEAR")["UNITS_SOLD"].idxmax()
    ]
)

print(top_retailers)